In [1]:
! pip install peft librosa soundfile gradio
! pip install git+https://github.com/huggingface/transformers
! pip install -U bitsandbytes>=0.46.1

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-qd8xtruq
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-qd8xtruq
  Resolved https://github.com/huggingface/transformers to commit a553395766001116a719c82870171f8d6b458c98
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11381798 sha256=59578940af070dffc505d6f38376dc30a2e849fb56809b15a120ec9e86a0916f
  Stored in directory: /tmp/pip-ephem-wheel-cache-1eo640y2/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
from peft import PeftModel
from transformers import BitsAndBytesConfig, AutoProcessor, Qwen2AudioForConditionalGeneration

# load base model
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Or "fp4"
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True
)

base_model = Qwen2AudioForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B", quantization_config=quantization_config,
                                                           device_map="auto", trust_remote_code=True)

temp = Qwen2AudioForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B", quantization_config=quantization_config,
                                                        device_map="auto", trust_remote_code=True)
lora_model = PeftModel.from_pretrained(temp, "KaileyM/qwen2audio-lora-music-descriptor", device_map="auto")

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B" ,trust_remote_code=True)

base_model.eval()
lora_model.eval()

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2AudioForConditionalGeneration(
      (audio_tower): Qwen2AudioEncoder(
        (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 1280)
        (layers): ModuleList(
          (0-31): 32 x Qwen2AudioEncoderLayer(
            (self_attn): Qwen2AudioAttention(
              (k_proj): Linear4bit(in_features=1280, out_features=1280, bias=False)
              (v_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1280, out_features=1280, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1280, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                 

In [4]:
import librosa
# inference generator
def describe_music(audio, model):
  prompt = "<|audio_bos|><|AUDIO|><|audio_eos|>Describe the music thoroughly:"

  inputs = processor(text=prompt, audio=audio, sampling_rate=16000, padding=False, return_tensors="pt")

  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

  # remove prompt tokens from generation
  generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

  description = processor.decode(
    generated_ids, skip_special_tokens=True)

  return description


In [5]:
def compare_outputs(audio_file):
  # load audio from file, limiting to 30 seconds for efficiency and for processor limit
  audio, _ = librosa.load(audio_file, duration=30.0, sr=16000)

  original = describe_music(audio, base_model)[0]

  fine_tuned = describe_music(audio, lora_model)[0]

  return original, fine_tuned


In [6]:
# app demo
import gradio as gr

with gr.Blocks() as MuDesc:
    gr.Markdown("# <center>MusicDescriptor</center>")
    gr.Markdown("<center>Upload an audio file or record your own to generate a description of the music.</center>")

    audio_input = gr.Audio(type="filepath")

    submit_btn = gr.Button("Generate Descriptions")

    with gr.Row():
      with gr.Column():
        gr.Markdown("## Original")
        output_orig = gr.Textbox(lines=10, label="Original")
      with gr.Column():
        gr.Markdown("## Fine-tuned")
        output_ft = gr.Textbox(lines=10, label="Fine-tuned")

    submit_btn.click(
        fn=compare_outputs,
        inputs=audio_input,
        outputs=[output_orig, output_ft]
    )

MuDesc.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://561dea25d1e8599ac5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
